In [1]:
import numpy as np

rng = np.random.default_rng(7)
n = 10000
sex   = rng.integers(0, 2, n)          # 性别
age5  = rng.integers(0, 16, n)         # 年龄（5 岁一组，0-79）
zip2  = rng.integers(0, 50, n)         # 邮编前两位（50 个区域）
month = rng.integers(0, 12, n)         # 就诊月份
dept  = rng.integers(0, 8, n)          # 初诊科室

attrs = [("性别", sex), ("年龄段", age5), ("邮编前2位", zip2),
         ("就诊月份", month), ("初诊科室", dept)]

cols = []
print(f"{'k':>2} {'added':<8} {'n_cells':>8} {'max_class':>9} {'unique_rate':>11}")
for k, (name, col) in enumerate(attrs, start=1):
    cols.append(col)
    mat = np.column_stack(cols)
    _, counts = np.unique(mat, axis=0, return_counts=True)
    uniq = np.mean(counts == 1)
    print(f"{k:>2} {name:<8} {len(counts):>8} {counts.max():>9} {uniq:>11.3f}")

 k added     n_cells max_class unique_rate
 1 性别              2      5074       0.000
 2 年龄段            32       349       0.000
 3 邮编前2位        1596        17       0.013
 4 就诊月份         7786         6       0.761
 5 初诊科室         9687         3       0.969


In [2]:
import numpy as np

rng = np.random.default_rng(2026)
lo, hi, sigma, n0, reps = 0.0, 10.0, 2.0, 1000, 2000
mu = 5.0

def dp_mean(x, eps):
    sens = (hi - lo) / len(x)          # 均值的 l1 敏感度
    return np.clip(x, lo, hi).mean() + rng.laplace(0.0, sens / eps)

def required_n(eps, v):
    """解 sigma^2/n + 2(u-l)^2/(eps^2 n^2) = v 的正根，v 为目标总方差"""
    a = 2.0 * (hi - lo) ** 2 / eps ** 2
    return (sigma ** 2 + np.sqrt(sigma ** 4 + 4 * v * a)) / (2 * v)

v0 = sigma ** 2 / n0                    # 无隐私约束时 n=1000 的抽样方差
base_hw = 1.96 * np.sqrt(v0)
print(f"baseline: n0={n0}, var={v0:.5f}, 95% half-width={base_hw:.4f}")
print(f"{'eps':>5} {'var_ratio@n0':>12} {'n_required':>11} {'sim_95hw':>9}")
for eps in (0.1, 0.5, 2.0):
    n_star = int(np.ceil(required_n(eps, v0)))
    errs = np.empty(reps)
    for r in range(reps):
        x = rng.normal(mu, sigma, n_star)
        errs[r] = dp_mean(x, eps) - mu
    hw = np.percentile(np.abs(errs), 95)
    ratio = (2 * (hi - lo) ** 2 / (eps ** 2 * n0 ** 2)) / v0
    print(f"{eps:>5} {ratio:>12.3f} {n_star:>11} {hw:>9.4f}")

baseline: n0=1000, var=0.00400, 95% half-width=0.1240
  eps var_ratio@n0  n_required  sim_95hw
  0.1        5.000        2792    0.1285
  0.5        0.200        1171    0.1212
  2.0        0.013        1013    0.1235


In [3]:
import numpy as np

def rdp_to_dp(eps_rdp: float, alpha: float, delta: float) -> float:
    """RDP 到近似 DP 的标准换算：eps = eps_rdp + log(1/delta)/(alpha-1)"""
    return eps_rdp + np.log(1.0 / delta) / (alpha - 1.0)

def eps_from_rdp_curve(rdp_curve, delta=1e-5, alphas=None):
    """rdp_curve: dict{alpha: eps_rdp}（已含 k 次组合的累加）；返回最小的换算 eps"""
    if alphas is None:
        alphas = np.arange(1.1, 64.0, 0.1)
    return min(rdp_to_dp(rdp_curve[a], a, delta) for a in alphas)

# 高斯机制在采样率 q、噪声乘子 sigma、步数 T 下的 RDP 曲线（示意，实际请用库实现）
def gaussian_rdp(q, sigma, T, alphas):
    return {a: T * q**2 * a / (2 * sigma**2) for a in alphas}

alphas = np.arange(1.1, 64.0, 0.1)
curve = gaussian_rdp(q=0.01, sigma=1.1, T=3000, alphas=alphas)
print("eps =", round(eps_from_rdp_curve(curve, delta=1e-5, alphas=alphas), 3))

eps = 2.513


In [ ]:
# import torch
# from opacus import PrivacyEngine

# model, optimizer, loader = build_model(), torch.optim.SGD(...), make_loader()

# privacy_engine = PrivacyEngine()
# model, optimizer, loader = privacy_engine.make_private_with_epsilon(
#     module=model,
#     optimizer=optimizer,
#     data_loader=loader,
#     epochs=20,
#     target_epsilon=8.0,     # 目标总预算，具体取值依数据敏感度与机构要求而定
#     target_delta=1e-5,      # 通常取远小于 1/n
#     max_grad_norm=1.0,      # 逐样本梯度裁剪范数 C
# )

# for x, y in loader:
#     optimizer.zero_grad()
#     loss = torch.nn.functional.cross_entropy(model(x), y)
#     loss.backward()
#     optimizer.step()

# eps = privacy_engine.get_epsilon(delta=1e-5)   # 训练后核对实际消耗
# print(f"achieved: eps={eps:.2f}, delta=1e-5")

# # 无框架时：Laplace 机制的均值发布（演示定义本身）
# import numpy as np
# def dp_mean(x, lo, hi, epsilon):
#     n = len(x)
#     sens = (hi - lo) / n
#     noise = np.random.laplace(loc=0.0, scale=sens / epsilon)
#     return float(np.clip(x, lo, hi).mean() + noise), 2 * (sens / epsilon) ** 2

In [5]:
import numpy as np

rng = np.random.default_rng(3)
d = 150
w_true = rng.normal(0, 1, d)

def make_data(m):
    X = rng.normal(0, 1, (m, d))
    logits = X @ w_true + rng.normal(0, 1.0, m)   # 标签噪声：贝叶斯误差不可消除
    y = (logits > 0).astype(float)
    return X, y

Xte, yte = make_data(2000)                        # 非成员池

def fit_logistic(X, y, lr=0.5, epochs=3000):
    n = len(X)
    Xb = np.hstack([X, np.ones((n, 1))])
    theta = np.zeros(d + 1)
    for _ in range(epochs):
        p = 1.0 / (1.0 + np.exp(-np.clip(Xb @ theta, -30, 30)))
        grad = Xb.T @ (p - y) / n
        theta -= lr * grad
    return theta

def per_nll(theta, X, y):
    """逐样本交叉熵（数值稳定），即攻击者可得的逐样本损失"""
    Xb = np.hstack([X, np.ones((len(X), 1))])
    z = np.clip(Xb @ theta, -30, 30)
    return np.logaddexp(0.0, z) - y * z

def auc(scores, members):
    """秩和法 AUC：分数越大越像成员；0.5 等于随机猜测"""
    order = np.argsort(scores)
    ranks = np.empty(len(scores))
    ranks[order] = np.arange(1, len(scores) + 1)
    n1, n0 = members.sum(), (~members).sum()
    return (ranks[members].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

# 两种情形：d > n_train（可插值，记忆训练集）对比 d 远小于 n_train（泛化）
for n_train, lr, ep in [(100, 0.5, 3000), (5000, 0.5, 200)]:
    Xtr, ytr = make_data(n_train)
    theta = fit_logistic(Xtr, ytr, lr, ep)
    members = np.concatenate([np.ones(n_train, bool), np.zeros(len(Xte), bool)])
    scores = np.concatenate([-per_nll(theta, Xtr, ytr), -per_nll(theta, Xte, yte)])
    print(f"n_train={n_train}: train_NLL={per_nll(theta, Xtr, ytr).mean():.4f} "
          f"test_NLL={per_nll(theta, Xte, yte).mean():.4f} "
          f"attack_AUC={auc(scores, members):.3f}")

n_train=100: train_NLL=0.0007 test_NLL=1.0661 attack_AUC=0.777
n_train=5000: train_NLL=0.1171 test_NLL=0.1372 attack_AUC=0.509


In [ ]:
# import hashlib, json, pathlib, datetime

# LOG = pathlib.Path("audit/log.jsonl")
# PREV = pathlib.Path("audit/chain.head")

# def append_event(actor: str, action: str, obj: str, result: str, note: str = ""):
#     prev = PREV.read_text().strip() if PREV.exists() else "genesis"
#     ev = {"ts": datetime.datetime.now(datetime.timezone.utc).isoformat(),
#           "actor": actor, "action": action, "object": obj,
#           "result": result, "note": note, "prev": prev}
#     body = json.dumps({k: v for k, v in ev.items() if k != "prev"},
#                       ensure_ascii=False, sort_keys=True)
#     ev["hash"] = hashlib.sha256((prev + body).encode()).hexdigest()
#     with LOG.open("a", encoding="utf-8") as f:
#         f.write(json.dumps(ev, ensure_ascii=False, sort_keys=True) + "\n")
#     PREV.write_text(ev["hash"])
#     return ev["hash"]

# # 完整性校验：从头重算哈希链，任一行被改动都会导致校验失败
# def verify_chain():
#     head, prev = "genesis", "genesis"
#     for line in LOG.read_text(encoding="utf-8").splitlines():
#         ev = json.loads(line)
#         body = json.dumps({k: v for k, v in ev.items() if k not in ("prev", "hash")},
#                           ensure_ascii=False, sort_keys=True)
#         if ev["prev"] != prev or ev["hash"] != hashlib.sha256((prev + body).encode()).hexdigest():
#             return False, ev
#         prev = ev["hash"]; head = ev["hash"]
#     return True, head